# Honest versus naive: one benchmark, two information sets

A fixed autoregressive benchmark (order by BIC inside each training window) is estimated at every forecast origin twice: on the vintage actually available then (honest) and on final data (naive). Both are scored against the same target, the latest-vintage growth of the target period. The gap is the flattery built into final-data evaluation.

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from realtime_forecast.vintages import load_rtdsm_matrix
from realtime_forecast.revisions import (
    growth_revision_frame, revision_stats, revision_stats_by_phase,
    noise_vs_news, vintage_coverage,
)
pd.set_option("display.width", 120)
from realtime_forecast.forecast import run_evaluation, score, diebold_mariano, block_bootstrap_gap
from realtime_forecast.bridge import run_bridge_evaluation


## CPI (monthly origins, 2000-01 to 2025-12 vintages)

In [2]:
cpi = matrices_cpi = load_rtdsm_matrix('../data/raw/pcpiMvMd.xlsx', 'M')
origins = [c for c in cpi.columns if pd.Period('2000-01', freq='M') <= c <= pd.Period('2025-12', freq='M')]
fc_cpi = run_evaluation(cpi, origins, max_p=12, min_train=60)
score(fc_cpi), diebold_mariano(fc_cpi), block_bootstrap_gap(fc_cpi, block=12)

({'n_origins': 310,
  'realtime_rmse': 0.27843288473002253,
  'realtime_mae': 0.1967725608781082,
  'naive_rmse': 0.27293891591767727,
  'naive_mae': 0.18926659202426166,
  'rmse_gap_naive_minus_realtime': -0.005493968812345262},
 {'n': 310,
  'mean_loss_diff_naive_minus_realtime': -0.003029219476765114,
  'dm_statistic': -2.2667352277810897,
  'dm_p_value': 0.02409737592312733,
  'lag': 1},
 {'reps': 2000,
  'block': 12,
  'seed': 20261110,
  'gap_p05': -0.010257141295227874,
  'gap_p50': -0.00577331399239589,
  'gap_p95': -0.001097110750401727,
  'gap_ci90_low': -0.010257141295227874,
  'gap_ci90_high': -0.001097110750401727,
  'share_of_reps_gap_negative': 0.9805})

## PPI (monthly origins, 2000-01 to 2025-12 vintages)

In [3]:
ppi = load_rtdsm_matrix('../data/raw/pppiMvMd.xlsx', 'M')
origins_p = [c for c in ppi.columns if pd.Period('2000-01', freq='M') <= c <= pd.Period('2025-12', freq='M')]
fc_ppi = run_evaluation(ppi, origins_p, max_p=12, min_train=60)
score(fc_ppi), diebold_mariano(fc_ppi), block_bootstrap_gap(fc_ppi, block=12)

({'n_origins': 312,
  'realtime_rmse': 0.7778638284103069,
  'realtime_mae': 0.5669985547176787,
  'naive_rmse': 0.7689375980525014,
  'naive_mae': 0.5579960139047541,
  'rmse_gap_naive_minus_realtime': -0.008926230357805487},
 {'n': 312,
  'mean_loss_diff_naive_minus_realtime': -0.01380710585038904,
  'dm_statistic': -0.8306819563882295,
  'dm_p_value': 0.40679056879039344,
  'lag': 1},
 {'reps': 2000,
  'block': 12,
  'seed': 20261110,
  'gap_p05': -0.02732414777868416,
  'gap_p50': -0.010093906760252902,
  'gap_p95': 0.012059658858829592,
  'gap_ci90_low': -0.02732414777868416,
  'gap_ci90_high': 0.012059658858829592,
  'share_of_reps_gap_negative': 0.793})

## PCE prices, autoregressive arm (quarterly origins)

In [4]:
pce = load_rtdsm_matrix('../data/raw/pconQvQd.xlsx', 'Q')
origins_q = [c for c in pce.columns if pd.Period('2000Q1', freq='Q') <= c <= pd.Period('2025Q4', freq='Q')]
fc_pce = run_evaluation(pce, origins_q, max_p=4, min_train=40)
score(fc_pce), diebold_mariano(fc_pce), block_bootstrap_gap(fc_pce, block=8)

({'n_origins': 104,
  'realtime_rmse': 0.4243523962716471,
  'realtime_mae': 0.2798194087774395,
  'naive_rmse': 0.41280111422382443,
  'naive_mae': 0.27831271191776297,
  'rmse_gap_naive_minus_realtime': -0.011551282047822686},
 {'n': 104,
  'mean_loss_diff_naive_minus_realtime': -0.009670196317058066,
  'dm_statistic': -0.8618479812968742,
  'dm_p_value': 0.3907732394203607,
  'lag': 1},
 {'reps': 2000,
  'block': 8,
  'seed': 20261110,
  'gap_p05': -0.02366001417683759,
  'gap_p50': -0.011324663607139351,
  'gap_p95': 0.003635083584804751,
  'gap_ci90_low': -0.02366001417683759,
  'gap_ci90_high': 0.003635083584804751,
  'share_of_reps_gap_negative': 0.899})

## PCE bridge specification

The bridge forecasts the quarter's PCE inflation from the first month's CPI and PPI inflation (the only monthly data published by the middle month of the quarter, the aligned vintage) plus the previous quarter's PCE inflation. Same two arms.

In [5]:
bridge = run_bridge_evaluation(pce, cpi, ppi, origins_q, min_train=40)
score(bridge), diebold_mariano(bridge), block_bootstrap_gap(bridge, block=8)

({'n_origins': 67,
  'realtime_rmse': 0.29577984324649875,
  'realtime_mae': 0.21109859638331097,
  'naive_rmse': 0.25931171410007203,
  'naive_mae': 0.20380924881016746,
  'rmse_gap_naive_minus_realtime': -0.03646812914642672},
 {'n': 67,
  'mean_loss_diff_naive_minus_realtime': -0.0202431506014059,
  'dm_statistic': -0.8493947673039032,
  'dm_p_value': 0.3987322224173002,
  'lag': 1},
 {'reps': 2000,
  'block': 8,
  'seed': 20261110,
  'gap_p05': -0.09292878950092893,
  'gap_p50': -0.03522703329161572,
  'gap_p95': 0.02782761625436176,
  'gap_ci90_low': -0.09292878950092893,
  'gap_ci90_high': 0.02782761625436176,
  'share_of_reps_gap_negative': 0.826})

## The leakage gate, demonstrated

The engine asserts per origin that its training column equals the as-of vintage column. Corrupting every later vintage leaves the real-time forecasts of earlier origins unchanged; the test suite pins both properties.

In [6]:
from realtime_forecast.synthetic import make_synthetic_matrix
syn, planted = make_synthetic_matrix()
origins_s = list(syn.columns[90:96])
clean = run_evaluation(syn, origins_s, max_p=4, min_train=60)
corrupt = syn.copy()
later = [c for c in corrupt.columns if c > origins_s[-1]]
corrupt.loc[:, later] = (corrupt.loc[:, later] * 7.0).to_numpy()
dirty = run_evaluation(corrupt, origins_s, max_p=4, min_train=60)
(clean['realtime_forecast'].to_numpy() == dirty['realtime_forecast'].to_numpy()).all()

np.True_